# Data Collection and Cleaning

In [1]:
## Importing the dataset
import pandas as pd
df = pd.read_csv(r"C:\Users\Admin\Downloads\churn-bigml-80.csv")
print(df.head())

  State  Account length  Area code International plan Voice mail plan  \
0    KS             128        415                 No             Yes   
1    OH             107        415                 No             Yes   
2    NJ             137        415                 No              No   
3    OH              84        408                Yes              No   
4    OK              75        415                Yes              No   

   Number vmail messages  Total day minutes  Total day calls  \
0                     25              265.1              110   
1                     26              161.6              123   
2                      0              243.4              114   
3                      0              299.4               71   
4                      0              166.7              113   

   Total day charge  Total eve minutes  Total eve calls  Total eve charge  \
0             45.07              197.4               99             16.78   
1             27.47   

In [9]:
## Data cleaning and checking of missing values and outliers

import pandas as pd
import numpy as np

def clean_churn_bigml_80_dataset(filepath):
  df = pd.read_csv(filepath)

  # Standardize column names
  df.columns = [col.strip() for col in df.columns]

  # 1. Check and handle missing values
  missing_count = df.isnull().sum().sum()
  if missing_count > 0:
    for col in df.select_dtypes(include=[np.number]).columns:
      df[col] = df[col].fillna(df[col].median())
    for col in df.select_dtypes(include=["object", "bool"]).columns:
      if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

  # 2. Flag extreme numerical outliers via IQR (without dropping valid high-usage rows)
  numeric_cols = [
      c
      for c in df.select_dtypes(include=[np.number]).columns
      if c != "Churn_binary"
  ]
  outlier_flags = pd.Series(False, index=df.index)
  for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr > 0:
      col_outliers = (df[col] < (q1 - 3 * iqr)) | (df[col] > (q3 + 3 * iqr))
      outlier_flags = outlier_flags | col_outliers
  df["Extreme_Outlier_Flag"] = outlier_flags.astype(int)

  # 3. Create binary churn target safely
  if "Churn" in df.columns and "Churn_binary" not in df.columns:
    df["Churn_binary"] = (
        df["Churn"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    )

  # 4. Drop exact duplicate rows
  df = df.drop_duplicates().reset_index(drop=True)

  return df


# Run cleaning and save for your portfolio repo
cleaned_df = clean_churn_bigml_80_dataset(
    r"C:\Users\Admin\Downloads\churn-bigml-80.csv"
)

if cleaned_df is not None:
  print(
      f"Cleaned shape: {cleaned_df.shape} | Extreme outliers flagged:"
      f" {cleaned_df['Extreme_Outlier_Flag'].sum()}"
  )
  cleaned_df.to_csv("churn_cleaned.csv", index=False)

print("\n=== 1. DATASET INFO (Columns & Data Types) ===")
df.info()

print("\n=== 2. BASIC STATISTICAL SUMMARY ===")
display(df.describe(include='all'))


Cleaned shape: (2666, 22) | Extreme outliers flagged: 38

=== 1. DATASET INFO (Columns & Data Types) ===
<class 'pandas.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   str    
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   str    
 4   Voice mail plan         2666 non-null   str    
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13 

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
count,2666,2666.000000,2666.000000,2666,2666,2666.000000,2666.00000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666.000000,2666
unique,51,NaN,NaN,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,WV,NaN,NaN,No,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,88,NaN,NaN,2396,1933,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2278
mean,NaN,100.620405,437.438860,NaN,NaN,8.021755,179.48162,100.310203,30.512404,200.386159,100.023631,17.033072,201.168942,100.106152,9.052689,10.237022,4.467367,2.764490,1.562641,NaN
std,NaN,39.563974,42.521018,NaN,NaN,13.612277,54.21035,19.988162,9.215733,50.951515,20.161445,4.330864,50.780323,19.418459,2.285120,2.788349,2.456195,0.752812,1.311236,NaN
min,NaN,1.000000,408.000000,NaN,NaN,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,43.700000,33.000000,1.970000,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,73.000000,408.000000,NaN,NaN,0.000000,143.40000,87.000000,24.380000,165.300000,87.000000,14.050000,166.925000,87.000000,7.512500,8.500000,3.000000,2.300000,1.000000,NaN
50%,NaN,100.000000,415.000000,NaN,NaN,0.000000,179.95000,101.000000,30.590000,200.900000,100.000000,17.080000,201.150000,100.000000,9.050000,10.200000,4.000000,2.750000,1.000000,NaN
75%,NaN,127.000000,510.000000,NaN,NaN,19.000000,215.90000,114.000000,36.700000,235.100000,114.000000,19.980000,236.475000,113.000000,10.640000,12.100000,6.000000,3.270000,2.000000,NaN


In [10]:
## Standardize and normalize data
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

# 1. Load cleaned churn dataset 
df = pd.read_csv(r"C:\Users\Admin\Downloads\churn-bigml-80.csv")
df.columns = [col.strip() for col in df.columns]

if "Churn_binary" not in df.columns:
  df["Churn_binary"] = (
      df["Churn"]
      .astype(str)
      .str.strip()
      .str.lower()
      .map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
  )

numeric_features = [
    "Account length",
    "Customer service calls",
    "Total day minutes",
    "Total eve minutes",
]
categorical_features = ["International plan"]

X = df[numeric_features + categorical_features]
y = df["Churn_binary"]

# 2. Split before fitting any scaler
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Standardize numeric features (switch to MinMaxScaler() if normalization is preferred)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

# 4. Bundle preprocessing + classifier into a single pipeline
clf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

clf_pipeline.fit(X_train, y_train)
y_prob = clf_pipeline.predict_proba(X_test)[:, 1]
y_pred = clf_pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# Save the full pipeline (scaler + encoder + model) for batch inference
joblib.dump(clf_pipeline, "churn_pipeline_scaled.pkl")

              precision    recall  f1-score   support

           0       0.86      0.96      0.91       456
           1       0.33      0.12      0.17        78

    accuracy                           0.84       534
   macro avg       0.60      0.54      0.54       534
weighted avg       0.79      0.84      0.80       534

ROC-AUC: 0.7618


['churn_pipeline_scaled.pkl']